# Raman: Renishaw WiRe and B&W Tek BTC655N

Raman spectra are `SpectroscopyData` too, on the same `wavenumber` axis, which
for Raman holds the Raman shift (Stokes positive). `ldp.RamanData` is the reader
namespace. The files are anatase TiO2 test fixtures under
`tests/spectroscopy/raman/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lab_data_parser as ldp

In [ ]:
RAMAN = Path("..") / "tests" / "spectroscopy" / "raman"
RENISHAW = RAMAN / "renishaw"
BTC655N = RAMAN / "btc655n"
sorted(p.name for p in RENISHAW.iterdir())

## Renishaw `.wdf`

The binary WiRe file carries the excitation wavelength, the acquisition time and
the header fields, so nothing has to be passed by hand.

In [ ]:
raman_532 = ldp.RamanData.from_renishaw_wdf(RENISHAW / "TiO2_532_1acc_01.wdf")
raman_785 = ldp.RamanData.from_renishaw_wdf(RENISHAW / "TiO2_785_1acc_01.wdf")
for spectrum in (raman_532, raman_785):
    print(spectrum.technique, spectrum.excitation_wavelength_nm, "nm,", spectrum.shape, spectrum.data_type)

## Renishaw text export

The two-column `.txt` export has no header, so the excitation wavelength is given.

In [ ]:
raman_532_txt = ldp.RamanData.from_renishaw_txt(RENISHAW / "TiO2_532_1acc_01.txt", excitation_wavelength_nm=532)
raman_532_txt.shape

## B&W Tek BTC655N export

The handheld spectrometer exports wavelength in nm; the reader converts it to the
Raman shift with the laser wavelength written in the file header.

In [ ]:
raman_btc = ldp.RamanData.from_btc655n_export(BTC655N / "test_TiO2.txt")
print(raman_btc.excitation_wavelength_nm, "nm,", raman_btc.shape)

## Compare the instruments

Cut the same shift window and normalise every spectrum to its maximum. The
anatase bands sit near 144, 397, 516 and 639 cm^-1.

In [ ]:
spectra = {
    "renishaw, 532 nm": raman_532,
    "renishaw, 785 nm": raman_785,
    "btc655n": raman_btc,
}
fig, ax = plt.subplots(figsize=(7, 4))
for offset, (label, spectrum) in enumerate(spectra.items()):
    processed = spectrum.select_shift_range(100, 800).normalise_max()
    ax.plot(processed.shift_per_cm, np.ravel(processed.values) + offset, label=label)
for band in (144, 397, 516, 639):
    ax.axvline(band, color="grey", lw=0.5, ls=":")
ax.set_xlabel(raman_532.axis_label("cm^-1"))
ax.set_ylabel("normalised intensity / –")
ax.legend()
plt.show()

## Other units

The shift can be read as an energy, and the scattered light as a wavelength
(which needs the excitation wavelength).

In [ ]:
print(raman_532.axis_label("meV"), raman_532.axis("meV")[[0, -1]])
print(raman_532.axis_label("nm"), raman_532.axis("nm")[[0, -1]])